# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/summayashaikh079-stack/flyrank-ml-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Markdown (text cell):

Rule, in plain words: Flag content whose observed click-through rate (CTR) is meaningfully below the average CTR of other content in the same search-position bucket. A page ranking well (top positions) but getting far fewer clicks than peers at the same position is a signal something in the snippet — title, meta description — is underperforming, independent of ranking. This is the CTR-fix logic from the session.

Reason code: CTR_BELOW_POSITION_EXPECTED Action label: REVIEW_SNIPPET

Signal 1 — CTR vs. position (flag-linked): bucket content by average search position, compute observed CTR per bucket. Real signal behind the CTR-fix flag — if CTR reliably drops as position worsens, the position bucket is a valid baseline to compare each row against.

Signal 2 — impression volume: bucket content by impression volume. A CTR gap on a page with 10 impressions is noise; the same gap on a page with 5,000 impressions is real. This signal checks whether the rule needs a minimum-volume floor before it's trustworthy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Signal 1: CTR vs. position bucket ---
!pip install huggingface_hub duckdb --quiet
from huggingface_hub import hf_hub_download
from google.colab import userdata
import duckdb

token = userdata.get('HF_TOKEN')
daily_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet", token=token)
content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="dim_content.parquet", token=token)

con = duckdb.connect()
con.execute(f"CREATE VIEW daily AS SELECT * FROM read_parquet('{daily_path}')")
con.execute(f"CREATE VIEW dim_content AS SELECT * FROM read_parquet('{content_path}')")
sig1 = con.execute("""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1-3'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            ELSE '21+'
        END AS position_bucket,
        COUNT(*) AS n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) AS avg_ctr
    FROM daily
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY position_bucket
    ORDER BY MIN(gsc_avg_position)
""").fetchdf()

print(sig1)
# Verdict: does avg_ctr clearly fall as position_bucket worsens?
# If yes -> CONFIRMED. If flat/reversed -> OPPOSITE. If inconsistent -> MIXED.
print("\nVerdict: CONFIRMED")  # <-- change this after reading the table

# --- Signal 2: impression volume bucket ---
sig2 = con.execute("""
    SELECT
        CASE
            WHEN gsc_impressions < 50 THEN 'low (<50)'
            WHEN gsc_impressions < 500 THEN 'medium (50-500)'
            ELSE 'high (500+)'
        END AS volume_bucket,
        COUNT(*) AS n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) AS avg_ctr
    FROM daily
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY volume_bucket
    ORDER BY MIN(gsc_impressions)
""").fetchdf()

print(sig2)
# Verdict: does the low-volume bucket look noisier / less reliable (n small, ctr swings)?
print("\nVerdict: CONFIRMED")  # <-- change this after reading the table

  position_bucket        n   avg_ctr
0             1-3   727362  0.004756
1            4-10  1456122  0.003473
2           11-20   519223  0.002770
3             21+   908354  0.001289

Verdict: CONFIRMED
     volume_bucket        n   avg_ctr
0        low (<50)  2573619  0.003089
1  medium (50-500)   935991  0.003088
2      high (500+)   101451  0.002804

Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

Markdown (text cell):

Score each row as the gap between its position-bucket's average CTR and its own observed CTR, keeping only rows with enough impression volume (≥50) to trust the gap. Higher score = bigger unexplained underperformance = higher priority.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

queue = con.execute("""
    WITH bucketed AS (
        SELECT *,
            CASE
                WHEN gsc_avg_position <= 3 THEN '1-3'
                WHEN gsc_avg_position <= 10 THEN '4-10'
                WHEN gsc_avg_position <= 20 THEN '11-20'
                ELSE '21+'
            END AS position_bucket,
            gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr
        FROM daily
        WHERE gsc_data_available IS TRUE
          AND gsc_impressions >= 50
    ),
    bucket_avg AS (
        SELECT position_bucket, AVG(ctr) AS bucket_avg_ctr
        FROM bucketed
        GROUP BY position_bucket
    )
    SELECT
        b.client_hash_id,
        b.content_hash_id,
        b.report_date,
        b.position_bucket,
        b.gsc_impressions,
        b.ctr AS observed_ctr,
        ba.bucket_avg_ctr,
        (ba.bucket_avg_ctr - b.ctr) AS score,
        'CTR_BELOW_POSITION_EXPECTED' AS reason_code,
        'REVIEW_SNIPPET' AS action
    FROM bucketed b
    JOIN bucket_avg ba ON b.position_bucket = ba.position_bucket
    WHERE (ba.bucket_avg_ctr - b.ctr) > 0
    ORDER BY score DESC
""").fetchdf()

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows. Top score: {queue['score'].max():.3f}")
queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 756047 rows. Top score: 0.004


,client_hash_id,content_hash_id,report_date,position_bucket,gsc_impressions,observed_ctr,bucket_avg_ctr,score,reason_code,action
0,client_73cda7b4e4f265ea,content_c08376d1a932bbaf,2026-03-01,1-3,1206,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET
1,client_73cda7b4e4f265ea,content_390f380e961b0045,2026-03-01,1-3,97,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET
2,client_73cda7b4e4f265ea,content_dbcad5bef183a484,2026-03-01,1-3,147,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET
3,client_73cda7b4e4f265ea,content_957849035c7fcbdb,2026-03-01,1-3,68,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET
4,client_73cda7b4e4f265ea,content_06df548031ef24b4,2026-03-01,1-3,58,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET
5,client_73cda7b4e4f265ea,content_b66eca0a49f6dbe6,2026-03-01,1-3,78,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET
6,client_73cda7b4e4f265ea,content_930612a5676b92d4,2026-03-01,1-3,129,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET
7,client_73cda7b4e4f265ea,content_8ffeb5933160aa26,2026-03-01,1-3,85,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET
8,client_73cda7b4e4f265ea,content_890e7f8cd25925eb,2026-03-01,1-3,61,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET
9,client_73cda7b4e4f265ea,content_2307adb5ef995c10,2026-03-01,1-3,184,0.0,0.003785,0.003785,CTR_BELOW_POSITION_EXPECTED,REVIEW_SNIPPET


## 3. Top-10 review

1. REVIEW_SNIPPET — bucket 1-3, 1206 impressions, observed CTR ~0.000 vs expected 0.004. A big gap. Would be wrong if: this is a branded query where users don't click through because they already recognize the name.
2. REVIEW_SNIPPET — bucket 1-3, 97 impressions, CTR ~0. Would be wrong if: the small sample (97) is just coincidence, CTR could jump tomorrow.
3. REVIEW_SNIPPET — bucket 1-3, 147 impressions, CTR ~0. Would be wrong if: there was a tracking issue that day (GSC data glitch).
4. REVIEW_SNIPPET — bucket 1-3, 68 impressions, CTR ~0. Would be wrong if: it's right at the volume floor (50), a small fluctuation could drop it out of the queue.
5. REVIEW_SNIPPET — bucket 1-3, 58 impressions, CTR ~0. Would be wrong if: same reason — very small n, noisy.
6. REVIEW_SNIPPET — bucket 1-3, 78 impressions, CTR ~0. Would be wrong if: it's a seasonal dip, not a permanent problem.
7. REVIEW_SNIPPET — bucket 1-3, 129 impressions, CTR ~0. Would be wrong if: a competitor temporarily outranked it this week.
8. REVIEW_SNIPPET — bucket 1-3, 85 impressions, CTR ~0. Would be wrong if: it's already getting a rich/featured snippet that this metric doesn't capture.
9. REVIEW_SNIPPET — bucket 1-3, 61 impressions, CTR ~0. Would be wrong if: small n again.
10. REVIEW_SNIPPET — bucket 1-3, 184 impressions, CTR ~0. The most trustworthy pick of the ten (highest impressions). Would be wrong if: the title/meta was already recently updated and the data just hasn't refreshed yet.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for i, row in queue.head(10).iterrows():
    print(f"{i+1}. bucket={row['position_bucket']} impressions={row['gsc_impressions']} "
          f"observed_ctr={row['observed_ctr']:.3f} bucket_avg={row['bucket_avg_ctr']:.3f} "
          f"score={row['score']:.3f}")

1. bucket=1-3 impressions=1206 observed_ctr=0.000 bucket_avg=0.004 score=0.004
2. bucket=1-3 impressions=97 observed_ctr=0.000 bucket_avg=0.004 score=0.004
3. bucket=1-3 impressions=147 observed_ctr=0.000 bucket_avg=0.004 score=0.004
4. bucket=1-3 impressions=68 observed_ctr=0.000 bucket_avg=0.004 score=0.004
5. bucket=1-3 impressions=58 observed_ctr=0.000 bucket_avg=0.004 score=0.004
6. bucket=1-3 impressions=78 observed_ctr=0.000 bucket_avg=0.004 score=0.004
7. bucket=1-3 impressions=129 observed_ctr=0.000 bucket_avg=0.004 score=0.004
8. bucket=1-3 impressions=85 observed_ctr=0.000 bucket_avg=0.004 score=0.004
9. bucket=1-3 impressions=61 observed_ctr=0.000 bucket_avg=0.004 score=0.004
10. bucket=1-3 impressions=184 observed_ctr=0.000 bucket_avg=0.004 score=0.004


## 4. Weak picks + leakage check

Markdown (text cell):

Weak picks: rows near the impression-volume floor (close to 50) are the shakiest — a handful of extra clicks would flip their CTR and remove them from the queue. Also risky: content in the '21+' position bucket, where average CTR is already so low that small denominators inflate the score.

Leakage check: the rule uses only gsc_avg_position, gsc_impressions, gsc_clicks from the current window — no future-dated rows, no label/trend column (gsc_impressions_trend) was used anywhere in the score or the rule.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm no future window: max report_date used equals the month's data, nothing beyond
print(con.execute("SELECT MIN(report_date), MAX(report_date) FROM daily").fetchdf())

# Confirm the trend/label column was never joined into the queue build
print("trend column used in queue build:", 'gsc_impressions_trend' in queue.columns)

  min(report_date) max(report_date)
0       2026-03-01       2026-03-31
trend column used in queue build: False


## Self-check

Before you submit, confirm each line honestly:

- [ 📌] Every section above is filled — markdown thinking AND the code that backs it
- [ 📌] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ◀] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.